In [1]:
import sys
import torch
import os
from transformers import AutoTokenizer, GPT2Tokenizer
import pickle
import pandas as pd
import platform
from torch.nn import functional as F
import tqdm


In [2]:
#Set Tokenizer Root as the folder that contains different tokenizer folders 
# Options - (relevant ones) - 
#   babylm_full_bpe_8k - Tokenizer for 10M models, vocab size 8k
#   babylm_full_bpe_100M_8k - Tokenizer for 100M models, vocab size 8k 
#Model's Relevant details can be found in the Model Table in the database (in rundata.xlsx)

TOKENIZER_ROOT = r"/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/data" 


#Set OUT_ROOT as the folder where all the models checkpoints are stored. Current setup expects the within OUT_ROOT to have a model with a folder_name, which contains a ckpt.pt file. ckpt.pt also contains the relevant model config details that get saved, so it is not explicitly required. To pick the right model, look into the database or rundata.xlsx file and pick the right model name (folder name) from OutputFolderName column(Database) or output_folder_name(from rundata.xlsx). 


MODELS_ROOT = r'/media/abishekthamma/Backup Plus/Projects/masters_thesis/ss-llm/nanoGPT/output_dump'

ROOT_ROOT = r'/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT'  

In [3]:


def load_model(out_dir, device="cuda"):
    """
    Loads a pre-trained GPT model from a checkpoint file.

    Args:
        out_dir (str): The directory where the checkpoint file is located.
        device (torch.device): The device to load the model onto.

    Returns:
        GPT: The loaded GPT model.

    Raises:
        FileNotFoundError: If the checkpoint file is not found.
    """
    ckpt_path = os.path.join(MODELS_ROOT, out_dir, 'ckpt.pt')
    print(f"Loading model from {ckpt_path}")
    # NANOGPT_ROOT = str(Path(__file__).parents[4])

    # Add if condition to check if inside server and if is, then add the path correctly. Default is local for now
    sys.path.append(ROOT_ROOT)
    from model import GPT, GPTConfig

    checkpoint = torch.load(ckpt_path, map_location=device)

    # Backward compatibility for new model args for QKV and FFW Adjustments
    if checkpoint["model_args"].get("wm_decay_length", None) is None:
        # wm_decay_length = block_size
        checkpoint["model_args"]["wm_decay_length"] = checkpoint["model_args"]["block_size"]
    # Setting head size as 3 times n_embd if not set already
    if checkpoint['model_args'].get('head_size_qkv', None) is None:
        checkpoint['model_args']['head_size_qkv'] = checkpoint['model_args']['n_embd']

    if checkpoint["model_args"].get("ffw_dim", None) is None:
        checkpoint["model_args"]["ffw_dim"] = 4 * checkpoint["model_args"]["n_embd"]

    # print(checkpoint['model_args'])
    gptconf = GPTConfig(**checkpoint['model_args'])

    load_model = GPT(gptconf)

    state_dict = checkpoint['model']
    unwanted_prefix = '_orig_mod.'
    for k, v in list(state_dict.items()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)

    load_model.load_state_dict(state_dict)
    load_model.eval()

    load_model = load_model.to(device)

    return load_model


def load_tokenizer(data_dir):
    """
    Load tokenizer for natural stories evaluation.

    Args:
        data_dir (str): The directory path where the tokenizer data is stored.

    Returns:
        tokenizer (Tokenizer): The loaded tokenizer object.

    Raises:
        NotImplementedError: If stoi/itos is not supported or found.

    """
    data_dir = os.path.join(TOKENIZER_ROOT, data_dir)
    meta_path = os.path.join(data_dir, "meta.pkl")
    load_meta = os.path.exists(meta_path)

    if load_meta:
        with open(meta_path, 'rb') as f:
            meta = pickle.load(f)
        if meta.get("custom_tokenizer", False):
            print(f"Loading custom tokenizer from {data_dir}")
            tokenizer = AutoTokenizer.from_pretrained(data_dir, use_fast=False)
        else:
            if meta.get("stoi", False):
                raise NotImplementedError("stoi/itos not supported yet")
            else:
                raise NotImplementedError("No stoi/itos found")
    else:
        print("No meta.pkl found")
        raise NotImplementedError("No meta.pkl found")

    if not tokenizer.eos_token:
        tokenizer.add_special_tokens({"eos_token": "</s>"})
    if not tokenizer.pad_token:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "left"  # Add if needed?
    return tokenizer


def load_model_tokenizer(out_dir, data_dir, device="cuda"):
    model = load_model(out_dir, device)
    tokenizer = load_tokenizer(data_dir)
    return model, tokenizer




In [4]:
out_dir = "out-babylm_full_bpe_8k-6x6-mask_ee2000_em01-6849723"
data_dir = "babylm_full_bpe_8k"

model, tokenizer = load_model_tokenizer(out_dir, data_dir)


Loading model from /media/abishekthamma/Backup Plus/Projects/masters_thesis/ss-llm/nanoGPT/output_dump/out-babylm_full_bpe_8k-6x6-mask_ee2000_em01-6849723/ckpt.pt


/tmp/ipykernel_246591/1603345888.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(ckpt_path, map_location=device)


Setting flash to False because wm_mask is enabled
Setting flash to False because wm_mask is enabled
Setting flash to False because wm_mask is enabled
Setting flash to False because wm_mask is enabled
Setting flash to False because wm_mask is enabled
Setting flash to False because wm_mask is enabled
number of parameters: 13.69M
Loading custom tokenizer from /home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/data/babylm_full_bpe_8k


In [5]:
model.eval()
model.to("cuda")

sample_text = "The quick brown fox jumped over the lazy dog"
# Encode the input text
input_ids = tokenizer.encode(sample_text, return_tensors="pt").to("cuda")

print([(i, repr(tokenizer.decode(i))) for i in input_ids[0]]) 

#Get Model Hidden States
with torch.no_grad():
    outputs = model(input_ids, hidden_states=True) #If you want Loss, pass expected tokens with target = tokens_to_predict


print("Logits: ", outputs[0].shape)
print("Loss", outputs[1])
print("Hidden States: ", [{f"Layer {i}": outputs[2][i].shape} for i in range(len(outputs[2]))])

[(tensor(251, device='cuda:0'), "'the'"), (tensor(1865, device='cuda:0'), "' quick'"), (tensor(2936, device='cuda:0'), "' brown'"), (tensor(4617, device='cuda:0'), "' fox'"), (tensor(7126, device='cuda:0'), "' jumped'"), (tensor(530, device='cuda:0'), "' over'"), (tensor(173, device='cuda:0'), "' the'"), (tensor(201, device='cuda:0'), "' l'"), (tensor(985, device='cuda:0'), "'az'"), (tensor(63, device='cuda:0'), "'y'"), (tensor(1767, device='cuda:0'), "' dog'")]
Logits:  torch.Size([1, 1, 8000])
Loss None
Hidden States:  [{'Layer 0': torch.Size([1, 11, 384])}, {'Layer 1': torch.Size([1, 11, 384])}, {'Layer 2': torch.Size([1, 11, 384])}, {'Layer 3': torch.Size([1, 11, 384])}, {'Layer 4': torch.Size([1, 11, 384])}, {'Layer 5': torch.Size([1, 11, 384])}, {'Layer 6': torch.Size([1, 11, 384])}]
